# Lab Session 1 — From the Unstructured Web to Structured Entities

**Domain:** Michael Jackson 


1. **Phase 1** : Web Crawling & nettoyage du HTML
2. **Phase 2.1** : Named Entity Recognition (NER)
3. **Phase 2.2** : Extraction de relations par Dependency Parsing





On utilise fr_dep_news_trf au lieu de en_core_web_lg
Toutes nos sources sont en français donc utiliser un modèle anglais produit des entités erronées 


In [1]:
!pip install trafilatura spacy pandas httpx
!python -m spacy download fr_dep_news_trf
!python -m spacy download fr_core_news_lg


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached https://github.com/explosion/spacy-models/releases/download/fr_dep_news_trf-3.8.0/fr_dep_news_trf-3.8.0-py3-none-any.whl (397.7 MB)
[+] Download and installation successful
You can now load the package via spacy.load('fr_dep_news_trf')



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/571.8 MB ? eta -:--:--
      ------------------------------------- 10.5/571.8 MB 59.3 MB/s eta 0:00:10
     - ------------------------------------ 21.0/571.8 MB 57.6 MB/s eta 0:00:10
     -- ----------------------------------- 30.9/571.8 MB 50.3 MB/s eta 0:00:11
     -- ----------------------------------- 41.7/571.8 MB 50.0 MB/s eta 0:00:11
     --- ---------------------------------- 52.4/571.8 MB 51.8 MB/s eta 0:00:11
     ---- --------------------------------- 64.0/571.8 MB 50.6 MB/s eta 0:00:11
     ----- -------------------------------- 76.0/571.8 MB 51.2 MB/s eta 0:00:10
     ----- -------------------------------- 87.0/571.8 MB 51.3 MB/s eta 0:00:10
     ------ ------------------------------- 97.5/571.8 MB 51.0 MB/s eta 0:00:10
     ------- ----------------------------- 109.3/571.8 MB 51.4 MB/s eta 0:00:09
     ------- ----------------------------- 119.8/571.8 MB 51.1 MB/s eta 0:00:09
     -------- ---------------------------- 130.


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Phase 1 — Web Crawling & Nettoyage
### 1.1 Sélection des URLs sources

10 pages biographiques 



In [2]:
URLS = [
    "https://fr.wikipedia.org/wiki/Michael_Jackson",
    "https://www.larousse.fr/encyclopedie/personnage/Michael_Jackson/155320",
    "https://www.nrj.fr/artistes/michael-jackson/biographie",
    "https://www.universalis.fr/encyclopedie/michael-jackson/",
    "https://www.morning-femina.fr/michael-jackson/",
    "https://www.linternaute.fr/musique/biographie/1777510-michael-jackson-chansons-cultes-vie-et-polemiques-biographie-du-roi-de-la-pop/",
    "https://www.nostalgie.fr/artistes/michael-jackson/biographie",
    "http://www.musicomania.ca/michaeljackson/biographie.htm",
    "https://www.nofi.media/2014/10/michael-jackson-3-2/1519",
    "https://www.cultivonsnous.fr/michael-jackson-histoire-et-biographie-de-jackson/",
]

print(f"{len(URLS)} URLs sources définies.")


10 URLs sources définies.


### 1.2 téléchargement et d'extraction

**`trafilatura`** extrait le contenu éditorial principal d'une page HTML en supprimant menus, publicités et footers.  



In [3]:
import trafilatura
import time

def fetch_and_extract(url):
    """
    Télécharge une URL et extrait le texte principal avec trafilatura.
    Retourne le texte nettoyé, ou None en cas d'échec.
    """
    print(f"\n  Téléchargement : {url}")
    downloaded = trafilatura.fetch_url(url)
    if downloaded is None:
        print("  ✗ Échec du téléchargement")
        return None
    print("  ✓ Téléchargement réussi")

    text = trafilatura.extract(downloaded)
    if text is None:
        print("  ✗ Extraction du texte échouée")
        return None

    print(f"  ✓ {len(text)} caractères extraits")
    return text


### onb vérifie la pertinence de la page

Une page avec moins de 500 mots n'apportera pas suffisamment d'entités exploitables


In [4]:
def is_useful(text, min_words=500):
    """
    Retourne True si le texte contient au moins min_words mots.
    """
    word_count = len(text.split())
    if word_count < min_words:
        print(f"  ✗ Page ignorée : {word_count} mots (minimum : {min_words})")
        return False
    print(f"  ✓ Page utile : {word_count} mots")
    return True


### Pipeline complet de crawling


In [5]:
results = []

for url in URLS:
    text = fetch_and_extract(url)
    if text is None:
        time.sleep(1)
        continue
    if not is_useful(text):
        time.sleep(1)
        continue
    results.append({'url': url, 'text': text})
    print("  → Page ajoutée au dataset")
    time.sleep(1)  # Respect éthique du rate limit

print(f"\n{len(results)} pages conservées sur {len(URLS)} tentées.")



  Téléchargement : https://fr.wikipedia.org/wiki/Michael_Jackson
  ✓ Téléchargement réussi
  ✓ 147148 caractères extraits
  ✓ Page utile : 24641 mots
  → Page ajoutée au dataset

  Téléchargement : https://www.larousse.fr/encyclopedie/personnage/Michael_Jackson/155320
  ✓ Téléchargement réussi
  ✓ 14558 caractères extraits
  ✓ Page utile : 2554 mots
  → Page ajoutée au dataset

  Téléchargement : https://www.nrj.fr/artistes/michael-jackson/biographie
  ✓ Téléchargement réussi
  ✓ 7665 caractères extraits
  ✓ Page utile : 1299 mots
  → Page ajoutée au dataset

  Téléchargement : https://www.universalis.fr/encyclopedie/michael-jackson/
  ✓ Téléchargement réussi
  ✓ 5670 caractères extraits
  ✓ Page utile : 946 mots
  → Page ajoutée au dataset

  Téléchargement : https://www.morning-femina.fr/michael-jackson/
  ✓ Téléchargement réussi
  ✓ 6122 caractères extraits
  ✓ Page utile : 1094 mots
  → Page ajoutée au dataset

  Téléchargement : https://www.linternaute.fr/musique/biographie/17775

### Sauvegarde au format JSONL

**JSONL (JSON Lines)** : un objet JSON par ligne.  


In [6]:
import json

OUTPUT_FILE = "crawler_output.jsonl"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"{len(results)} pages sauvegardées dans '{OUTPUT_FILE}'")


10 pages sauvegardées dans 'crawler_output.jsonl'


---
## Phase 2 — Extraction d'Information
### 2.1 Named Entity Recognition (NER)

#### Chargement du modèle spaCy français

On utilise `fr_dep_news_trf` qui est un modèle Transformer entraîné sur de la presse française.

Labels utilisés par le modèle français : `PER` (personnes), `ORG` (organisations), `LOC` (lieux), `MISC` (divers)


In [7]:
import spacy
import json

nlp = spacy.load("fr_core_news_lg")
print(f"Modèle chargé : {nlp.meta['name']} (langue : {nlp.meta['lang']})")

# Rechargement des textes depuis le fichier JSONL
texts = []
with open("crawler_output.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        texts.append(json.loads(line))

print(f"{len(texts)} documents chargés.")


Modèle chargé : core_news_lg (langue : fr)
10 documents chargés.


#### Extraction des entités nommées


In [8]:
# Labels cibles (nomenclature modèle français)
TARGET_LABELS = {"PER", "ORG", "LOC", "MISC"}

# Correspondance vers noms standardisés pour la suite du projet
LABEL_MAP = {"PER": "PERSON", "ORG": "ORG", "LOC": "GPE", "MISC": "MISC"}

entities_raw = []

for doc_data in texts:
    url  = doc_data["url"]
    text = doc_data["text"]
    print(f"\n  NER : {url}")

    # Limite à 100 000 caractères pour éviter les dépassements mémoire
    doc = nlp(text[:100_000])

    count = 0
    for ent in doc.ents:
        if ent.label_ in TARGET_LABELS:
            entities_raw.append({
                "entity": ent.text.strip(),
                "label":  LABEL_MAP[ent.label_],
                "url":    url,
            })
            count += 1
    print(f"  → {count} entités extraites")

print(f"\nTotal brut : {len(entities_raw)} entités")



  NER : https://fr.wikipedia.org/wiki/Michael_Jackson
  → 1327 entités extraites

  NER : https://www.larousse.fr/encyclopedie/personnage/Michael_Jackson/155320
  → 228 entités extraites

  NER : https://www.nrj.fr/artistes/michael-jackson/biographie
  → 124 entités extraites

  NER : https://www.universalis.fr/encyclopedie/michael-jackson/
  → 111 entités extraites

  NER : https://www.morning-femina.fr/michael-jackson/
  → 102 entités extraites

  NER : https://www.linternaute.fr/musique/biographie/1777510-michael-jackson-chansons-cultes-vie-et-polemiques-biographie-du-roi-de-la-pop/
  → 172 entités extraites

  NER : https://www.nostalgie.fr/artistes/michael-jackson/biographie
  → 124 entités extraites

  NER : http://www.musicomania.ca/michaeljackson/biographie.htm
  → 538 entités extraites

  NER : https://www.nofi.media/2014/10/michael-jackson-3-2/1519
  → 136 entités extraites

  NER : https://www.cultivonsnous.fr/michael-jackson-histoire-et-biographie-de-jackson/
  → 638 entit

#### filtrage des entités non pertinentes

On supprime :
- Entités trop courtes (< 3 caractères) 
- Entités purement numériques
- Entités trop longues (> 50 caractères) 
- Mots génériques issus des tableaux Wikipedia
- Doublons


In [9]:
import pandas as pd

GENERIC_WORDS = {"naissance", "décès", "activité", "instruments", "genre",
                 "surnom", "membre", "années", "label", "chanteur"}

def is_valid_entity(text):
    text = text.strip()
    if len(text) < 3:                      return False  
    if text.isnumeric():                   return False  
    if len(text) > 50:                     return False  
    if text.lower() in GENERIC_WORDS:      return False  
    return True

clean_entities = []
seen = set()

for e in entities_raw:
    key = (e["entity"].lower(), e["label"], e["url"])  # Dédoublonnage par page
    if key in seen or not is_valid_entity(e['entity']):
        continue
    seen.add(key)
    clean_entities.append(e)

df_entities = pd.DataFrame(clean_entities)
print(f"Entités après nettoyage : {len(df_entities)}")
print("\nRépartition par type :")
print(df_entities["label"].value_counts().to_string())
print("\nTop PERSON :", df_entities[df_entities["label"]=="PERSON"]["entity"].value_counts().head(8).index.tolist())
print("Top ORG    :", df_entities[df_entities["label"]=="ORG"]["entity"].value_counts().head(8).index.tolist())
print("Top GPE    :", df_entities[df_entities["label"]=="GPE"]["entity"].value_counts().head(8).index.tolist())


Entités après nettoyage : 2005

Répartition par type :
label
MISC      738
PERSON    726
GPE       273
ORG       268

Top PERSON : ['Jackson', 'Michael Jackson', 'Michael', 'Billie Jean', 'Quincy Jones', 'Jermaine', 'Elvis Presley', 'Berry Gordy']
Top ORG    : ['Motown', 'ABC', 'Jackson Five', 'les Jacksons', 'Sony', 'Triumph', 'Epic Records', 'Beatles']
Top GPE    : ['Indiana', 'Gary', 'Californie', 'États-Unis', 'Neverland', 'Paris', 'Jacksons', 'Los Angeles']


#### Sauvegarde des entités


In [10]:
df_entities.to_csv("entities_mJackson.csv", index=False)
print(f"{len(df_entities)} entités sauvegardées dans 'entities_mJackson.csv'")
df_entities.head(10)


2005 entités sauvegardées dans 'entities_mJackson.csv'


,entity,label,url
0,Michael Jackson,PERSON,https://fr.wikipedia.org/wiki/Michael_Jackson
1,The King of Pop,MISC,https://fr.wikipedia.org/wiki/Michael_Jackson
2,|---|---|,MISC,https://fr.wikipedia.org/wiki/Michael_Jackson
3,Michael Joseph Jackson[1,PERSON,https://fr.wikipedia.org/wiki/Michael_Jackson
4,Indiana,GPE,https://fr.wikipedia.org/wiki/Michael_Jackson
5,États-Unis,GPE,https://fr.wikipedia.org/wiki/Michael_Jackson
6,| Décès,PERSON,https://fr.wikipedia.org/wiki/Michael_Jackson
7,Los Angeles,GPE,https://fr.wikipedia.org/wiki/Michael_Jackson
8,Californie,GPE,https://fr.wikipedia.org/wiki/Michael_Jackson
9,Disco,ORG,https://fr.wikipedia.org/wiki/Michael_Jackson


---
### 2.2 Extraction de Relations par Dependency Parsing
on met sous la forme:
```Entité_A  --[nsubj]--> verbe <--[obj/dobj]--  Entité_B
```
→ Triplet : `(Entité_A, verbe_lemmatisé, Entité_B)`

exemple: 
*"Michael Jackson rejoint Epic Records en 1975."*  
→ `(Michael Jackson, rejoindre, Epic Records)`


In [11]:
def extract_triplets_from_text(text, nlp_model):
    triplets = []
    doc = nlp_model(text[:100_000])

    for sent in doc.sents:
        ents_in_sent = [e for e in sent.ents if e.label_ in TARGET_LABELS]
        if len(ents_in_sent) < 2:
            continue

       
        for token in sent:
            if token.pos_ not in ("VERB", "AUX"):
                continue

            subj_token = obj_token = None
            for child in token.children:
                if child.dep_ in ("nsubj", "nsubj:pass") and subj_token is None:
                    subj_token = child
                if child.dep_ in ("obj", "dobj", "obl", "xcomp", "attr") and obj_token is None:
                    obj_token = child

           
            if subj_token is None:
                for child in sent.root.children:
                    if child.dep_ in ("nsubj", "nsubj:pass"):
                        subj_token = child
                        break

            if subj_token is None or obj_token is None:
                continue

            subj_ent = obj_ent = None
            for ent in ents_in_sent:
                if ent.start <= subj_token.i < ent.end:
                    subj_ent = ent
                if ent.start <= obj_token.i < ent.end:
                    obj_ent = ent

            if subj_ent and obj_ent and subj_ent != obj_ent:
                triplets.append({
                    "subject":   subj_ent.text.strip(),
                    "predicate": token.lemma_.lower().replace(" ", "_"),
                    "object":    obj_ent.text.strip(),
                    "sentence":  sent.text.strip(),
                })

        # ── Approche 2 : Co-occurrence dans la même phrase ──
        # Toutes les paires d'entités dans la même phrase → is_related_to
        for i in range(len(ents_in_sent)):
            for j in range(i + 1, len(ents_in_sent)):
                e1 = ents_in_sent[i]
                e2 = ents_in_sent[j]
                triplets.append({
                    "subject":   e1.text.strip(),
                    "predicate": "is_related_to",
                    "object":    e2.text.strip(),
                    "sentence":  sent.text.strip(),
                })

    return triplets

#### Application sur toutes les pages


In [12]:
all_triplets = []

for page in texts:
    url  = page.get("url", "")
    text = page.get("text", "")
    if not text:
        continue
    print(f"\n  Extraction : {url}")
    triplets = extract_triplets_from_text(text, nlp)
    all_triplets.extend(triplets)
    print(f"  → {len(triplets)} triplets")

print(f"\nTotal brut : {len(all_triplets)} triplets")



  Extraction : https://fr.wikipedia.org/wiki/Michael_Jackson
  → 2861 triplets

  Extraction : https://www.larousse.fr/encyclopedie/personnage/Michael_Jackson/155320
  → 442 triplets

  Extraction : https://www.nrj.fr/artistes/michael-jackson/biographie
  → 362 triplets

  Extraction : https://www.universalis.fr/encyclopedie/michael-jackson/
  → 167 triplets

  Extraction : https://www.morning-femina.fr/michael-jackson/
  → 123 triplets

  Extraction : https://www.linternaute.fr/musique/biographie/1777510-michael-jackson-chansons-cultes-vie-et-polemiques-biographie-du-roi-de-la-pop/
  → 191 triplets

  Extraction : https://www.nostalgie.fr/artistes/michael-jackson/biographie
  → 362 triplets

  Extraction : http://www.musicomania.ca/michaeljackson/biographie.htm
  → 1300 triplets

  Extraction : https://www.nofi.media/2014/10/michael-jackson-3-2/1519
  → 159 triplets

  Extraction : https://www.cultivonsnous.fr/michael-jackson-histoire-et-biographie-de-jackson/
  → 1263 triplets

Tota

#### Nettoyage des triplets

Suppression des doublons, relations réflexives (sujet = objet) et prédicats vides.


In [13]:
df_triplets = pd.DataFrame(all_triplets)

# Dédoublonnage 
df_triplets = df_triplets.drop_duplicates(subset=["subject", "predicate", "object"])

# Suppression des relations réflexives
df_triplets = df_triplets[df_triplets['subject'] != df_triplets['object']]

# Suppression des prédicats trop courts 
df_triplets = df_triplets[df_triplets['predicate'].str.len() > 1]

print(f"Triplets après nettoyage : {len(df_triplets)}")
print("\nPrédicats les plus fréquents :")
print(df_triplets["predicate"].value_counts().head(15).to_string())
print("\nExemples de triplets :")
df_triplets[["subject", "predicate", "object"]].head(15)


Triplets après nettoyage : 5202

Prédicats les plus fréquents :
predicate
is_related_to    5156
sortir              4
surnommer           4
recevoir            2
participe           2
jurer               1
convoquer           1
signer              1
enregistrer         1
interpréter         1
visiter             1
terminer            1
diffus              1
quitte              1
persuader           1

Exemples de triplets :


,subject,predicate,object
0,The King of Pop,is_related_to,|---|---|
1,Indiana,is_related_to,États-Unis
2,Los Angeles,is_related_to,Californie
3,Los Angeles,is_related_to,États-Unis
4,Californie,is_related_to,États-Unis
5,Disco,is_related_to,new jack swing
6,Sammy Davis,is_related_to,Jr
7,Diana Ross,is_related_to,James Brown
8,Diana Ross,is_related_to,Fred Astaire
9,Diana Ross,is_related_to,Charlie Chaplin


#### Sauvegarde des triplets


In [14]:
df_triplets.to_csv("triplets_initiaux.csv", index=False)
print(f"{len(df_triplets)} triplets sauvegardés dans 'triplets_initiaux.csv'")


5202 triplets sauvegardés dans 'triplets_initiaux.csv'


---
## Statistiques Finales



In [15]:
df_t = pd.read_csv("triplets_initiaux.csv")
df_e = pd.read_csv("entities_mJackson.csv")

n_triplets  = len(df_t)
n_entities  = len(df_e["entity"].unique())
n_predicats = len(df_t["predicate"].unique())
n_sources   = len(df_e["url"].unique())

print("=" * 50)
print("STATISTIQUES FINALES — KB Initiale")
print("=" * 50)
print(f"  Triplets          : {n_triplets:>6}   (min requis : 100)")
print(f"  Entités uniques   : {n_entities:>6}   (min requis : 50)")
print(f"  Prédicats uniques : {n_predicats:>6}")
print(f"  Sources crawlées  : {n_sources:>6}")
print("=" * 50)
print(f"  {'OK' if n_triplets >= 100 else 'KO'} >= 100 triplets  ({n_triplets})")
print(f"  {'OK' if n_entities >= 50  else 'KO'} >= 50 entités   ({n_entities})")

print("\nFichiers produits :")
print("  -> crawler_output.jsonl    (textes bruts + URLs)")
print("  -> entities_mJackson.csv   (entités nommées)")
print("  -> triplets_initiaux.csv   (relations extraites)")


STATISTIQUES FINALES — KB Initiale
  Triplets          :   5202   (min requis : 100)
  Entités uniques   :   1136   (min requis : 50)
  Prédicats uniques :     39
  Sources crawlées  :     10
  OK >= 100 triplets  (5202)
  OK >= 50 entités   (1136)

Fichiers produits :
  -> crawler_output.jsonl    (textes bruts + URLs)
  -> entities_mJackson.csv   (entités nommées)
  -> triplets_initiaux.csv   (relations extraites)
